# BirdCLEF 2026: MobileNetV3 + Weakly-Supervised Event Mining

Цей ноутбук спеціально побудований **не як повторення** поточного `audio_lab2.ipynb` і **не як варіація EfficientNet-B0 + Focal Loss + cosine scheduler**.

## Що тут інше

- Ми об'єднуємо `train_audio` і `train_soundscapes_labels` в **єдиний multi-label training manifest** на 234 класи.
- Для змішаних soundscape не намагаємось робити повний source separation без сильних анотацій. Замість цього використовуємо **weakly-supervised event mining**:
  - будуємо `mel + PCEN`
  - знаходимо найбільш активні часові області
  - подаємо моделі **event-focused spectrogram crop**, а не весь фон однаково
- Backbone: **MobileNetV3-Small**, а не EfficientNet-B0.
- Loss: **Asymmetric Loss** для multi-label tagging, а не Focal Loss.
- Scheduler: **OneCycleLR**, а не cosine.
- Додаємо **auxiliary taxonomic head** на 5 груп (`Aves`, `Amphibia`, `Mammalia`, `Reptilia`, `Insecta`), щоб стабілізувати навчання на рідкісних видах.
- Версія в цьому файлі спеціально зроблена **lightweight**, щоб її було легше тренувати на одній GPU або навіть на Colab/Kaggle T4.

## Чому це має сенс

У `train.csv` є відносно чисті записи окремих видів, а в `train_soundscapes_labels.csv` є реальні змішані 5-секундні вікна. Для задачі з 234 класами це краще формулювати як **audio tagging / weakly supervised detection**, а не як звичайну single-label класифікацію.

In [1]:
# Якщо запускаєш поза Kaggle і пакетів бракує, розкоментуй:
# !pip install -q -r requirements.txt

import ast
import gc
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision

from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()

@dataclass
class CFG:
    data_root: str = "birdclef-2026"
    sr: int = 24000
    clip_seconds: float = 5.0
    event_crop_seconds: float = 3.0
    n_mels: int = 128
    fmin: int = 20
    fmax: int = 12000
    image_size: int = 160
    batch_size: int = 24
    num_workers: int = 0
    epochs: int = 5
    lr: float = 1e-3
    weight_decay: float = 1e-4
    fold: int = 0
    num_folds: int = 5
    soundscape_repeat: int = 2
    min_rating: float = 1.5
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_pretrained: bool = True
    event_focus_prob: float = 0.6
    valid_event_focus: bool = True
    freeze_backbone_epochs: int = 1

cfg = CFG()
ROOT = Path(cfg.data_root)
ROOT

/Users/khrystynakorets/Desktop/audio_lab2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('birdclef-2026')

In [2]:
train_df = pd.read_csv(ROOT / "train.csv")
soundscape_df = pd.read_csv(ROOT / "train_soundscapes_labels.csv")
taxonomy_df = pd.read_csv(ROOT / "taxonomy.csv")
sample_submission = pd.read_csv(ROOT / "sample_submission.csv")

print("train.csv:", train_df.shape)
print("train_soundscapes_labels.csv:", soundscape_df.shape)
print("taxonomy.csv:", taxonomy_df.shape)
print("sample_submission.csv:", sample_submission.shape)

print("\nTaxonomy by class:")
print(taxonomy_df["class_name"].value_counts())

train_unique = set(train_df["primary_label"].astype(str))
soundscape_unique = set(";".join(soundscape_df["primary_label"].astype(str)).split(";"))
taxonomy_unique = set(taxonomy_df["primary_label"].astype(str))

print("\nUnique labels in train_audio:", len(train_unique))
print("Unique labels in soundscapes:", len(soundscape_unique))
print("Union:", len(train_unique | soundscape_unique))
print("All taxonomy labels covered:", len(train_unique | soundscape_unique) == len(taxonomy_unique))

train.csv: (35549, 15)
train_soundscapes_labels.csv: (1478, 4)
taxonomy.csv: (234, 5)
sample_submission.csv: (3, 235)

Taxonomy by class:
class_name
Aves        162
Amphibia     35
Insecta      28
Mammalia      8
Reptilia      1
Name: count, dtype: int64

Unique labels in train_audio: 206
Unique labels in soundscapes: 75
Union: 234
All taxonomy labels covered: True


In [3]:
label_list = taxonomy_df["primary_label"].astype(str).tolist()
label_to_idx = {label: idx for idx, label in enumerate(label_list)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

class_names = sorted(taxonomy_df["class_name"].unique().tolist())
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
label_to_class = dict(zip(taxonomy_df["primary_label"].astype(str), taxonomy_df["class_name"]))

def parse_secondary_labels(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return [str(v) for v in parsed]
    except Exception:
        pass
    return []

def labels_to_target(labels):
    target = np.zeros(len(label_list), dtype=np.float32)
    for label in labels:
        if label in label_to_idx:
            target[label_to_idx[label]] = 1.0
    return target

def labels_to_group_target(labels):
    target = np.zeros(len(class_names), dtype=np.float32)
    for label in labels:
        group = label_to_class.get(label)
        if group is not None:
            target[class_to_idx[group]] = 1.0
    return target

print("Num species:", len(label_list))
print("Taxonomic groups:", class_names)

Num species: 234
Taxonomic groups: ['Amphibia', 'Aves', 'Insecta', 'Mammalia', 'Reptilia']


In [4]:
# 1) Clean / semi-clean clips from train_audio
clean_rows = []

filtered_train = train_df.copy()
filtered_train["rating"] = filtered_train["rating"].fillna(0).astype(float)
filtered_train = filtered_train[filtered_train["rating"] >= cfg.min_rating].reset_index(drop=True)

for row in filtered_train.itertuples(index=False):
    primary = str(row.primary_label)
    secondary = [lab for lab in parse_secondary_labels(row.secondary_labels) if lab in label_to_idx]
    labels = sorted(set([primary] + secondary))
    clean_rows.append(
        {
            "source_type": "train_audio",
            "audio_path": str(ROOT / "train_audio" / row.filename),
            "filename": row.filename,
            "segment_start": np.nan,
            "segment_end": np.nan,
            "labels": labels,
            "target": labels_to_target(labels),
            "group_target": labels_to_group_target(labels),
            "primary_label": primary,
            "rating": float(row.rating),
        }
    )

clean_manifest = pd.DataFrame(clean_rows)
clean_manifest["is_soundscape"] = 0

# 2) 5-second multi-label windows from train_soundscapes
sound_rows = []

def to_seconds(ts):
    h, m, s = ts.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

for row in soundscape_df.itertuples(index=False):
    labels = [lab for lab in str(row.primary_label).split(";") if lab in label_to_idx]
    sound_rows.append(
        {
            "source_type": "soundscape",
            "audio_path": str(ROOT / "train_soundscapes" / row.filename),
            "filename": row.filename,
            "segment_start": float(to_seconds(row.start)),
            "segment_end": float(to_seconds(row.end)),
            "labels": labels,
            "target": labels_to_target(labels),
            "group_target": labels_to_group_target(labels),
            "primary_label": labels[0],
            "rating": 5.0,
        }
    )

sound_manifest = pd.DataFrame(sound_rows)
sound_manifest["is_soundscape"] = 1

# Невелике oversampling soundscapes, бо це найцінніша частина для mixed-audio generalization
train_manifest = pd.concat(
    [clean_manifest] + [sound_manifest.copy() for _ in range(cfg.soundscape_repeat)],
    ignore_index=True,
)

print("Clean manifest:", clean_manifest.shape)
print("Soundscape manifest:", sound_manifest.shape)
print("Unified train manifest:", train_manifest.shape)
train_manifest.head()

Clean manifest: (22531, 11)
Soundscape manifest: (1478, 11)
Unified train manifest: (25487, 11)


,source_type,audio_path,filename,segment_start,segment_end,labels,target,group_target,primary_label,rating,is_soundscape
0,train_audio,birdclef-2026/train_audio/1595929/XC999239.ogg,1595929/XC999239.ogg,NaN,NaN,[1595929],"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0]",1595929,5.0,0
1,train_audio,birdclef-2026/train_audio/22930/XC940773.ogg,22930/XC940773.ogg,NaN,NaN,[22930],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0]",22930,3.0,0
2,train_audio,birdclef-2026/train_audio/22956/XC900618.ogg,22956/XC900618.ogg,NaN,NaN,[22956],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0]",22956,4.0,0
3,train_audio,birdclef-2026/train_audio/22961/XC1062603.ogg,22961/XC1062603.ogg,NaN,NaN,[22961],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0]",22961,5.0,0
4,train_audio,birdclef-2026/train_audio/22973/XC1062605.ogg,22973/XC1062605.ogg,NaN,NaN,[22973],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0]",22973,4.0,0


## Event Mining замість повного source separation

Для цього датасету в нас є слабкі анотації: ми знаємо, які види присутні у 5-секундному вікні, але не маємо точних покадрових меж кожного окремого звуку.

Тому практичніший підхід:

- перетворити аудіо в `mel spectrogram`
- застосувати `PCEN`, що добре приглушує стабільний фон і підсилює короткі вокалізації
- знайти найактивніший часовий центр
- обрізати спектрограму навколо цієї події

Це не ідеальна деміксація, але це хороший **weakly-supervised substitute** для задачі зі змішаними аудіо.

In [5]:
def load_audio_segment(path, start=None, end=None, sr=32000):
    waveform, source_sr = torchaudio.load(path)
    waveform = waveform.mean(dim=0)
    if source_sr != sr:
        waveform = torchaudio.functional.resample(waveform, source_sr, sr)

    if start is not None and end is not None:
        start_idx = int(start * sr)
        end_idx = int(end * sr)
        waveform = waveform[start_idx:end_idx]

    return waveform.numpy()

def pad_or_crop(waveform, target_length):
    if len(waveform) < target_length:
        pad = target_length - len(waveform)
        waveform = np.pad(waveform, (0, pad))
    elif len(waveform) > target_length:
        max_offset = len(waveform) - target_length
        offset = np.random.randint(0, max_offset + 1)
        waveform = waveform[offset:offset + target_length]
    return waveform

def build_pcen_mel(waveform, sr, n_mels, fmin, fmax):
    mel = librosa.feature.melspectrogram(
        y=waveform,
        sr=sr,
        n_fft=1024,
        hop_length=320,
        win_length=1024,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
        power=2.0,
    )
    pcen = librosa.pcen(mel * (2 ** 31), sr=sr, hop_length=320)
    pcen = pcen.astype(np.float32)
    return pcen

def event_focus_crop(spec, total_seconds=5.0, crop_seconds=3.0):
    time_energy = spec.mean(axis=0)
    smooth = np.convolve(time_energy, np.ones(5) / 5, mode="same")
    peak_idx = int(np.argmax(smooth))

    total_frames = spec.shape[1]
    crop_frames = max(8, int(total_frames * crop_seconds / total_seconds))
    left = max(0, peak_idx - crop_frames // 2)
    right = min(total_frames, left + crop_frames)
    left = max(0, right - crop_frames)

    cropped = spec[:, left:right]
    cropped = torch.tensor(cropped)[None, None, ...]
    resized = F.interpolate(cropped, size=(spec.shape[0], total_frames), mode="bilinear", align_corners=False)
    return resized[0, 0].numpy()

def normalize_spec(spec):
    spec = spec - spec.mean()
    spec = spec / (spec.std() + 1e-6)
    return spec

class BirdCLEFMultiLabelDataset(Dataset):
    def __init__(self, df, cfg, train_mode=True):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.train_mode = train_mode
        self.target_length = int(cfg.sr * cfg.clip_seconds)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row.audio_path

        if row.source_type == "soundscape":
            waveform = load_audio_segment(
                path,
                start=float(row.segment_start),
                end=float(row.segment_end),
                sr=self.cfg.sr,
            )
            waveform = pad_or_crop(waveform, self.target_length)
        else:
            waveform = load_audio_segment(path, sr=self.cfg.sr)
            waveform = pad_or_crop(waveform, self.target_length)

        spec = build_pcen_mel(
            waveform=waveform,
            sr=self.cfg.sr,
            n_mels=self.cfg.n_mels,
            fmin=self.cfg.fmin,
            fmax=self.cfg.fmax,
        )

        use_focus = self.train_mode and (np.random.rand() < self.cfg.event_focus_prob)
        if (row.source_type == "soundscape" and use_focus) or (not self.train_mode and self.cfg.valid_event_focus):
            spec = event_focus_crop(
                spec,
                total_seconds=self.cfg.clip_seconds,
                crop_seconds=self.cfg.event_crop_seconds,
            )

        spec = normalize_spec(spec)
        image = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.cfg.image_size, self.cfg.image_size),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        target = torch.tensor(row.target, dtype=torch.float32)
        group_target = torch.tensor(row.group_target, dtype=torch.float32)

        return {
            "image": image,
            "target": target,
            "group_target": group_target,
        }

In [6]:
class GeMPool(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        x = x.clamp(min=self.eps).pow(self.p)
        x = F.avg_pool2d(x, (x.size(-2), x.size(-1))).pow(1.0 / self.p)
        return x.flatten(1)

class BirdCLEFLightModel(nn.Module):
    def __init__(self, num_classes, num_groups, pretrained=True):
        super().__init__()
        weights = None
        if pretrained:
            try:
                weights = torchvision.models.MobileNet_V3_Small_Weights.DEFAULT
            except Exception:
                weights = None

        backbone = torchvision.models.mobilenet_v3_small(weights=weights)
        self.features = backbone.features
        self.pool = GeMPool()
        self.dropout = nn.Dropout(p=0.2)
        self.species_head = nn.Linear(576, num_classes)
        self.group_head = nn.Linear(576, num_groups)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.dropout(x)
        species_logits = self.species_head(x)
        group_logits = self.group_head(x)
        return species_logits, group_logits

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        pos_probs = probs
        neg_probs = 1 - probs

        if self.clip is not None and self.clip > 0:
            neg_probs = (neg_probs + self.clip).clamp(max=1)

        loss = targets * torch.log(pos_probs.clamp(min=self.eps))
        loss += (1 - targets) * torch.log(neg_probs.clamp(min=self.eps))

        pt = targets * pos_probs + (1 - targets) * neg_probs
        gamma = targets * self.gamma_pos + (1 - targets) * self.gamma_neg
        one_sided_w = torch.pow(1 - pt, gamma)

        return -(one_sided_w * loss).mean()

def lwlrap(truth, scores):
    truth = np.asarray(truth) > 0
    scores = np.asarray(scores)

    num_classes = truth.shape[1]
    precisions_for_samples_by_classes = []
    labels_per_class = truth.sum(axis=0)

    for k in range(num_classes):
        pos_mask = truth[:, k]
        if not pos_mask.any():
            precisions_for_samples_by_classes.append(np.zeros(0))
            continue

        class_scores = scores[:, k]
        order = np.argsort(-class_scores)
        class_truth = pos_mask[order]
        retrieved_cumulative = np.cumsum(class_truth)
        precision_at_hits = retrieved_cumulative[class_truth] / (1 + np.flatnonzero(class_truth))
        precisions_for_samples_by_classes.append(precision_at_hits)

    per_class_lwlrap = np.zeros(num_classes, dtype=np.float32)
    weights = labels_per_class.astype(np.float32)
    if weights.sum() == 0:
        return 0.0

    for k in range(num_classes):
        if labels_per_class[k] > 0:
            per_class_lwlrap[k] = precisions_for_samples_by_classes[k].mean()

    return float((per_class_lwlrap * weights / weights.sum()).sum())

species_criterion = AsymmetricLoss()
group_criterion = nn.BCEWithLogitsLoss()

model = BirdCLEFLightModel(
    num_classes=len(label_list),
    num_groups=len(class_names),
    pretrained=cfg.use_pretrained,
).to(cfg.device)

def set_backbone_trainable(model, trainable):
    for param in model.features.parameters():
        param.requires_grad = trainable

model

BirdCLEFLightModel(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(

In [7]:
clean_manifest["fold"] = -1
skf = StratifiedKFold(n_splits=cfg.num_folds, shuffle=True, random_state=SEED)
for fold, (_, val_idx) in enumerate(skf.split(clean_manifest, clean_manifest["primary_label"])):
    clean_manifest.loc[val_idx, "fold"] = fold

sound_manifest["fold"] = -1
gkf = GroupKFold(n_splits=min(cfg.num_folds, sound_manifest["filename"].nunique()))
for fold, (_, val_idx) in enumerate(gkf.split(sound_manifest, groups=sound_manifest["filename"])):
    sound_manifest.loc[val_idx, "fold"] = fold

train_manifest = pd.concat(
    [clean_manifest] + [sound_manifest.copy() for _ in range(cfg.soundscape_repeat)],
    ignore_index=True,
)

train_fold = train_manifest[train_manifest["fold"] != cfg.fold].reset_index(drop=True)
valid_fold = train_manifest[train_manifest["fold"] == cfg.fold].drop_duplicates(
    subset=["source_type", "filename", "segment_start", "segment_end"]
).reset_index(drop=True)

print("Train fold:", train_fold.shape)
print("Valid fold:", valid_fold.shape)
print(valid_fold["source_type"].value_counts())

# Class-aware sample weights: середнє inverse frequency по позитивних класах
class_freq = np.stack(train_fold["target"].values).sum(axis=0)
class_inv = 1.0 / np.clip(class_freq, 1, None)
sample_weights = []
for target in train_fold["target"].values:
    pos = np.where(target > 0)[0]
    sample_weights.append(float(class_inv[pos].mean()) if len(pos) else 1.0)

train_ds = BirdCLEFMultiLabelDataset(train_fold, cfg, train_mode=True)
valid_ds = BirdCLEFMultiLabelDataset(valid_fold, cfg, train_mode=False)

train_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    sampler=train_sampler,
    num_workers=cfg.num_workers,
    pin_memory=False,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=False,
)


Train fold: (20384, 12)
Valid fold: (4656, 12)
source_type
train_audio    4507
soundscape      149
Name: count, dtype: int64


/Users/khrystynakorets/Desktop/audio_lab2/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [9]:
set_backbone_trainable(model, False)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=cfg.lr,
    epochs=cfg.epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,
    anneal_strategy="linear",
)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.device == "cuda"))

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="train", leave=False):
        images = batch["image"].to(cfg.device)
        targets = batch["target"].to(cfg.device)
        group_targets = batch["group_target"].to(cfg.device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.device == "cuda")):
            logits, group_logits = model(images)
            loss_species = species_criterion(logits, targets)
            loss_group = group_criterion(group_logits, group_targets)
            loss = loss_species + 0.2 * loss_group

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / max(len(loader), 1)

@torch.no_grad()
def valid_one_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_probs = []

    for batch in tqdm(loader, desc="valid", leave=False):
        images = batch["image"].to(cfg.device)
        targets = batch["target"].to(cfg.device)
        group_targets = batch["group_target"].to(cfg.device)

        logits, group_logits = model(images)
        loss_species = species_criterion(logits, targets)
        loss_group = group_criterion(group_logits, group_targets)
        loss = loss_species + 0.2 * loss_group
        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    bin_preds = (all_probs >= 0.3).astype(np.float32)
    macro_f1 = f1_score(all_targets, bin_preds, average="macro", zero_division=0)
    score_lwlrap = lwlrap(all_targets, all_probs)

    return total_loss / max(len(loader), 1), macro_f1, score_lwlrap

best_score = -1.0
history = []

for epoch in range(cfg.epochs):
    if epoch == cfg.freeze_backbone_epochs:
        set_backbone_trainable(model, True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=cfg.lr,
            epochs=cfg.epochs - epoch,
            steps_per_epoch=len(train_loader),
            pct_start=0.15,
            anneal_strategy="linear",
        )

    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
    valid_loss, valid_f1, valid_lwlrap = valid_one_epoch(model, valid_loader)

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "valid_macro_f1": valid_f1,
            "valid_lwlrap": valid_lwlrap,
        }
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"valid_loss={valid_loss:.4f} | "
        f"macro_f1={valid_f1:.4f} | "
        f"lwlrap={valid_lwlrap:.4f}"
    )

    if valid_lwlrap > best_score:
        best_score = valid_lwlrap
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label_list": label_list,
                "class_names": class_names,
                "cfg": cfg.__dict__,
                "best_lwlrap": best_score,
            },
            "birdclef2026_light_event_mining_fold0.pth",
        )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

history_df = pd.DataFrame(history)
history_df

/var/folders/wf/9swpvl2n6f7_q7n364hc__bm0000gn/T/ipykernel_41884/1296716441.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.device == "cuda"))


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.11.0) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call last

In [ ]:
@torch.no_grad()
def predict_clip(model, audio_path, start_sec, end_sec, cfg):
    model.eval()

    waveform = load_audio_segment(audio_path, start=start_sec, end=end_sec, sr=cfg.sr)
    waveform = pad_or_crop(waveform, int(cfg.sr * cfg.clip_seconds))
    spec = build_pcen_mel(
        waveform=waveform,
        sr=cfg.sr,
        n_mels=cfg.n_mels,
        fmin=cfg.fmin,
        fmax=cfg.fmax,
    )
    spec = event_focus_crop(
        spec,
        total_seconds=cfg.clip_seconds,
        crop_seconds=cfg.event_crop_seconds,
    )
    spec = normalize_spec(spec)
    image = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1).unsqueeze(0).to(cfg.device)

    logits, _ = model(image)
    probs = torch.sigmoid(logits)[0].cpu().numpy()
    return probs

submission_cols = sample_submission.columns.tolist()[1:]

def build_submission(model, cfg):
    rows = []

    for row_id in tqdm(sample_submission["row_id"].tolist(), desc="submission"):
        parts = row_id.split("_")
        end_sec = int(parts[-1])
        base_filename = "_".join(parts[:-1]) + ".ogg"
        audio_path = ROOT / "test_soundscapes" / base_filename
        probs = predict_clip(model, audio_path, end_sec - 5, end_sec, cfg)

        row = {"row_id": row_id}
        row.update({label: float(probs[label_to_idx[label]]) for label in submission_cols})
        rows.append(row)

    return pd.DataFrame(rows)

# Коли будеш готова до інференсу:
# checkpoint = torch.load("birdclef2026_light_event_mining_fold0.pth", map_location=cfg.device)
# model.load_state_dict(checkpoint["model_state_dict"])
# submission = build_submission(model, cfg)
# submission.to_csv("submission.csv", index=False)
# submission.head()